# Insomnia Prediction — Ilaro, Ogun State, Nigeria

This notebook builds a complete machine learning pipeline that predicts whether a person is
likely to have **insomnia**, based on answers to 10 clinical sleep-screening questions plus a
few demographic details. The dataset is a synthetic sample reflecting the demographic and
lifestyle context of Lagos State / Ilaro, Ogun State, Nigeria.

**Pipeline overview:**
1. Load and explore the data
2. Clean and preprocess it (encoding + scaling)
3. Split into train/test sets
4. Train a Logistic Regression classifier
5. Evaluate the model
6. Inspect which features matter most
7. Save the trained model and scaler for deployment in the Streamlit app


## 1. Data Loading & Exploration

We start by loading `lagos_insomnia_dataset.csv` and getting a feel for its shape, data types,
summary statistics, and how balanced the two classes (insomnia vs. no insomnia) are.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Load the dataset
df = pd.read_csv("lagos_insomnia_dataset.csv")

print("Shape of dataset (rows, columns):", df.shape)
df.head()


In [ ]:
# Data types of each column
df.dtypes


In [ ]:
# Summary statistics for all columns (numeric + categorical)
df.describe(include="all")


### Class distribution

Here we check how many people in the dataset have insomnia (`1`) vs. do not (`0`), and
visualize it. A realistic screening dataset is usually imbalanced rather than perfectly
50/50 — we designed this dataset with roughly a 55% / 45% split.


In [ ]:
print(df["insomnia"].value_counts())
print()
print(df["insomnia"].value_counts(normalize=True).rename("proportion"))

plt.figure(figsize=(6, 4))
sns.countplot(x="insomnia", hue="insomnia", data=df,
              palette=["#4C72B0", "#DD8452"], legend=False)
plt.title("Class Distribution: Insomnia vs. No Insomnia")
plt.xlabel("Insomnia (0 = No, 1 = Yes)")
plt.ylabel("Number of People")
plt.show()


## 2. Preprocessing

Before we can train a model, we need to:
- Check for missing values
- Convert the "Yes"/"No" screening-question answers into 1/0
- One-hot encode multi-category columns (`gender`, `occupation`)
- Scale numeric features (`age`) with `StandardScaler` so the model isn't biased by scale


In [ ]:
# Check for missing (null) values in every column
df.isnull().sum()


In [ ]:
# The 9 Yes/No screening-question columns get simple binary encoding.
binary_cols = [
    "trouble_falling_asleep", "trouble_returning_to_sleep", "early_waking",
    "happens_often", "over_6_months", "enough_sleep", "daytime_impact",
    "daily_caffeine", "on_medication"
]
for col in binary_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

# feel_on_waking is a two-level categorical (Tired/Rested) -> label-encode as binary too
df["feel_on_waking"] = df["feel_on_waking"].map({"Tired": 1, "Rested": 0})

df[binary_cols + ["feel_on_waking"]].head()


In [ ]:
# gender and occupation are multi-category (or nominal) columns -> one-hot encode them
df_encoded = pd.get_dummies(df, columns=["gender", "occupation"], drop_first=True)

# Make sure one-hot columns are plain integers (0/1), not booleans
bool_cols = df_encoded.select_dtypes(include="bool").columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print("Shape after one-hot encoding:", df_encoded.shape)
df_encoded.head()


In [ ]:
# Separate features (X) and target (y)
feature_cols = [c for c in df_encoded.columns if c != "insomnia"]
X = df_encoded[feature_cols]
y = df_encoded["insomnia"]

# Scale the numeric column ('age') with StandardScaler.
# The binary/one-hot columns are already 0/1, so we only need to scale 'age'.
from sklearn.preprocessing import StandardScaler

numeric_cols = ["age"]
scaler = StandardScaler()

X_scaled = X.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])

X_scaled.head()


## 3. Train/Test Split

We hold out 20% of the data for testing, using `random_state=42` for reproducibility, and
stratify on the target so both sets keep roughly the same class balance.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)


## 4. Model Training

We fit a `LogisticRegression` model — a strong, interpretable baseline for binary
classification problems like this one.


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully.")


## 5. Evaluation

We evaluate the model on the held-out test set using accuracy, a full classification report
(precision, recall, F1-score for each class), and a confusion matrix.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["No Insomnia", "Insomnia"]))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Insomnia", "Insomnia"],
            yticklabels=["No Insomnia", "Insomnia"])
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix")
plt.show()


## 6. Feature Importance

Since Logistic Regression is a linear model, its coefficients tell us how strongly (and in
which direction) each feature pushes the prediction toward "insomnia." Positive coefficients
increase the predicted risk; negative coefficients decrease it.


In [ ]:
coefficients = pd.Series(model.coef_[0], index=X_train.columns).sort_values()

plt.figure(figsize=(9, 7))
coefficients.plot(kind="barh", color=["#DD8452" if v > 0 else "#4C72B0" for v in coefficients])
plt.title("Feature Importance (Logistic Regression Coefficients)")
plt.xlabel("Coefficient Value (impact on log-odds of insomnia)")
plt.tight_layout()
plt.show()

coefficients.sort_values(ascending=False)


## 7. Model Saving

Finally, we persist the trained model, the scaler, and the exact list of feature columns
(so the Streamlit app can rebuild the one-hot-encoded input row in the same order the model
expects) using `joblib`.


In [ ]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(list(X_train.columns), "feature_columns.pkl")

print("Saved: model.pkl, scaler.pkl, feature_columns.pkl")


## Summary

We loaded a synthetic insomnia-screening dataset for Ilaro, Ogun State, Nigeria, cleaned and
encoded it, trained a Logistic Regression classifier, evaluated its performance, and saved the
model artifacts (`model.pkl`, `scaler.pkl`, `feature_columns.pkl`) for use in the companion
Streamlit application (`app.py`).
